# Agentic Data Analysis Pipeline with Evaluation Framework

**Date:** May 2026  
**Environment:** Google Colab  
**Model:** claude-sonnet-4-6

---

### What this notebook does

This notebook builds and evaluates a Claude-powered agent that autonomously performs exploratory data analysis (EDA) on a real business dataset. Rather than using Claude as a text generator, it operates as a tool-calling agent in a closed reasoning loop — selecting statistical tests, executing code, observing results, and iterating until a set of evidenced findings can be synthesized.

The notebook is structured in two halves:

1. **Build** — dataset preparation, agent architecture, tool definitions, and a live agent run
2. **Evaluate** — a structured benchmark that scores the agent on statistical accuracy, hallucination rate, reasoning coherence, and test selection appropriateness — then documents fixes and shows before/after performance

---

> **Reproducibility note:** All agent runs are logged in full. Random seeds are set where applicable. Results may vary slightly across runs due to LLM non-determinism, but aggregate metrics over N=30 runs are stable.



---



---



## Install Dependencies & Libraries

In [1]:
# Install required packages
!pip install anthropic pandas scipy statsmodels matplotlib seaborn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 10.3 MB/s eta 0:00:00


In [2]:
import anthropic
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.stats.multitest as multitest
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings

warnings.filterwarnings("ignore")

print("All imports successful")

All imports successful


## API Key

In [3]:
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("Client initialized successfully")

Client initialized successfully




---



## Section 1: Dataset

### Objective
We need a dataset with known statistical properties so we can evaluate whether the agent finds what is actually there. A dataset without ground truth makes evaluation impossible — we'd have no way to distinguish a correct insight from a confident hallucination.

### Why UCI Bank Marketing
The [UCI Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing) is ideal for this project for four reasons:

1. **Mixed feature types** — numeric and categorical columns, which stress-tests the agent's test selection logic
2. **Known class imbalance** — the target variable (`y`: did the client subscribe?) skews ~88/12, a property the agent should detect
3. **Published ground truths** — the original Moro et al. (2014) paper documents which features are most predictive, giving us an external reference
4. **Real-world messiness** — age distributions, job categories, and contact durations all have properties that make naive statistical choices wrong

### Next Action
- Load the raw dataset
- Inspect its shape, types, and basic properties
- Understand what we are working with before the agent touches it

## Load in Data

In [4]:
# Load UCI Bank Marketing dataset directly from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"

import urllib.request
import zipfile
import io

# Download and extract
response = urllib.request.urlopen(url)
zip_data = zipfile.ZipFile(io.BytesIO(response.read()))
zip_data.extractall("/content/bank_data")

# Load the full dataset
df = pd.read_csv("/content/bank_data/bank-additional/bank-additional-full.csv", sep=";")

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

Shape: (41188, 21)

Columns: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


**Insight**

- 21 columns with 41188 rows, large enough dataset to analyze with real world application
- 'Y' target column is included
- Several categories to explore


## EDA - Initial Inspection & Quality Check

In [5]:
# Inspect data types and null counts
print("=== Data Types & Null Counts ===\n")
info = pd.DataFrame({
    "dtype": df.dtypes,
    "null_count": df.isnull().sum(),
    "null_pct": (df.isnull().sum() / len(df) * 100).round(2)
})

print(info)

=== Data Types & Null Counts ===

                  dtype  null_count  null_pct
age               int64           0       0.0
job              object           0       0.0
marital          object           0       0.0
education        object           0       0.0
default          object           0       0.0
housing          object           0       0.0
loan             object           0       0.0
contact          object           0       0.0
month            object           0       0.0
day_of_week      object           0       0.0
duration          int64           0       0.0
campaign          int64           0       0.0
pdays             int64           0       0.0
previous          int64           0       0.0
poutcome         object           0       0.0
emp.var.rate    float64           0       0.0
cons.price.idx  float64           0       0.0
cons.conf.idx   float64           0       0.0
euribor3m       float64           0       0.0
nr.employed     float64           0       0.0


**Insight**

- 0 nulls, unusual for real-world data. Needs further inspection.
- Y output column is object type, should be changed to int, binary 1/0

## Unknown / Missing Values

In [6]:
# Check for "unknown" encoded missing values in categorical columns
cat_cols = df.select_dtypes(include="object").columns.tolist()

print("=== 'unknown' counts in categorical columns ===\n")
for col in cat_cols:
    n = (df[col] == "unknown").sum()
    pct = (n / len(df) * 100).round(2)
    if n > 0:
        print(f"{col:<20} {n:>6} ({pct}%)")

print("\nDone.")

=== 'unknown' counts in categorical columns ===

job                     330 (0.8%)
marital                  80 (0.19%)
education              1731 (4.2%)
default                8597 (20.87%)
housing                 990 (2.4%)
loan                    990 (2.4%)

Done.


**Insight**

- "Unknown" entries make up 28% of the dataset.
- Default making 20% may need a different approach
- Manageable to handle missing data with furhter inspection
- Housing & loan have the same % of 'Unknow', could be a collection method issue

## Class Imbalance

In [7]:
# Check target variable class balance
print("=== Target Variable Distribution (y) ===\n")

counts = df["y"].value_counts()
pcts = df["y"].value_counts(normalize=True) * 100

balance = pd.DataFrame({
    "count": counts,
    "percentage": pcts.round(2)
})

print(balance)

=== Target Variable Distribution (y) ===

     count  percentage
y                     
no   36548       88.73
yes   4640       11.27


**Insight**

- Significant class imbalance at 88.73% / 11.27%.


**Challanges to address for Agent**

- When the agent compares feature distributions between subscribers and non-subscribers, the "yes" group has only 4,640 rows vs 36,548 "no" rows. Effect sizes will be meaningful but variance in the minority class will be higher.

- When the agent runs and reports on class balance, we can score it precisely



---



## Statistical Summary - Numeric Columns

In [8]:
# Summarize numeric columns
print("=== Numeric Column Summary ===\n")

numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
print(df[numeric_cols].describe().round(2))

=== Numeric Column Summary ===

            age  duration  campaign     pdays  previous  emp.var.rate  \
count  41188.00  41188.00  41188.00  41188.00  41188.00      41188.00   
mean      40.02    258.29      2.57    962.48      0.17          0.08   
std       10.42    259.28      2.77    186.91      0.49          1.57   
min       17.00      0.00      1.00      0.00      0.00         -3.40   
25%       32.00    102.00      1.00    999.00      0.00         -1.80   
50%       38.00    180.00      2.00    999.00      0.00          1.10   
75%       47.00    319.00      3.00    999.00      0.00          1.40   
max       98.00   4918.00     56.00    999.00      7.00          1.40   

       cons.price.idx  cons.conf.idx  euribor3m  nr.employed  
count        41188.00       41188.00   41188.00     41188.00  
mean            93.58         -40.50       3.62      5167.04  
std              0.58           4.63       1.73        72.25  
min             92.20         -50.80       0.63      4963.

**Insight**

- 'pdays' multiple 999 needs further understanding
- 'Duration' heavily right skewed
- 'Campaign' outlier problem
- 'Previous' multiple 0, meaning no previous contact



---



---



## Section 2: Ground Truth Injection

### Objective
Before the agent runs, we need to know what a correct analysis looks like. This is the foundation of the entire evaluation framework — without ground truths defined upfront, we have no objective basis for scoring the agent's output.

### The problem with evaluating EDA agents
Unlike classification models where accuracy is unambiguous, EDA outputs are narrative. An agent could produce fluent, confident, plausible-sounding insights that are statistically wrong — and without a pre-defined answer key, a reviewer might not catch it.

### Our approach
We inject three artificial signals directly into the dataset:

1. **A strong correlation** between two columns that did not previously correlate
2. **A group difference** — one job category will have a meaningfully different average call duration
3. **An outlier cluster** — a small group of rows with anomalous age values

These injections are documented here, locked in before the agent runs, and used as the scoring rubric. The agent must detect them. If it reports them without running the appropriate test, that is a hallucination. If it runs the wrong test, that is a test selection failure.

### What we do NOT change
The real structure of the dataset is preserved. We are adding signals on top, not replacing the underlying data.



---



## Inject Ground Truth for Testing


### **Challenge to address**

- When the agent analyzes the real Bank Marketing data and says "euribor3m correlates with duration" — is that correct?

- We don't actually know for certain without weeks of domain expertise. We can't score it objectively. And if the agent misses something, we can't tell whether that thing was genuinely there to be found.

- By injecting known signals ourselves, we flip the problem. Now we control exactly what's in the data. We know Injection 1 exists with r=0.871. We know Injection 2 produces a 137-second gap. We know Injection 3 added 150 outliers.

- The agent either finds them or it doesn't — no ambiguity, no domain expertise required to score it.

- This is the same principle used in security research — you don't evaluate a threat detection system by waiting for real attacks. You inject known threats and measure detection rate.



### **How we inject ground truths**

We inject three types of signals that map to the three most common EDA tasks — correlation detection, group comparison, and outlier identification. Each injection is designed to require a different statistical test, which stress-tests the agent's test selection logic across a range of scenarios.

| # | Signal type | Columns affected | What we inject | Expected test |
|---|-------------|-----------------|----------------|---------------|
| 1 | Correlation | `euribor3m` → `duration` | Strong linear relationship (r ~ 0.87) | Pearson correlation |
| 2 | Group difference | `job` == `"retired"` → `duration` | +180 seconds above group mean | Mann-Whitney U (skewed data) |
| 3 | Outlier cluster | `age` | 150 clients aged 85–98 | Distribution inspection + IQR |


### Three design rules we follow

1. **Inject into a copy, never the original.** `df_injected = df.copy()` preserves the raw data. The original `df` is never touched after this point.

2. **Set a random seed before every injection.** `np.random.seed(42)` ensures the exact same signals are produced on every run — essential for a reproducible benchmark.

3. **Verify each injection immediately after applying it.** We print the actual correlation, group means, and outlier count right after injection. If a signal didn't land as expected, we catch it here before the agent ever sees the data.

In [9]:
# Inject ground truths into a copy of the dataset
# Never modify the original df — always work from df_injected
df_injected = df.copy()

np.random.seed(42)

# --- Injection 1: Strong correlation ---
# Make euribor3m strongly predict duration (r ~ 0.7)
noise = np.random.normal(0, 30, len(df_injected))
df_injected["duration"] = (df_injected["euribor3m"] * 50 + 50 + noise).astype(int).clip(lower=0)

# --- Injection 2: Group difference ---
# Retired clients get significantly longer call durations
retired_mask = df_injected["job"] == "retired"
df_injected.loc[retired_mask, "duration"] = (
    df_injected.loc[retired_mask, "duration"] + 180
).clip(lower=0)

# --- Injection 3: Outlier cluster ---
# Inject 150 anomalously old clients (age 85-98)
outlier_idx = np.random.choice(df_injected.index, size=150, replace=False)
df_injected.loc[outlier_idx, "age"] = np.random.randint(85, 99, size=150)

print("=== Injections applied to df_injected ===\n")
print(f"Injection 1 — duration/euribor3m correlation")
r, p = stats.pearsonr(df_injected["euribor3m"], df_injected["duration"])
print(f"  Pearson r = {r:.3f}, p = {p:.2e}\n")

print(f"Injection 2 — retired group duration difference")
retired = df_injected.loc[retired_mask, "duration"]
others = df_injected.loc[~retired_mask, "duration"]
print(f"  Retired mean = {retired.mean():.1f}s, Others mean = {others.mean():.1f}s\n")

print(f"Injection 3 — age outlier cluster")
print(f"  Clients aged 85+: {(df_injected['age'] >= 85).sum()}")

=== Injections applied to df_injected ===

Injection 1 — duration/euribor3m correlation
  Pearson r = 0.871, p = 0.00e+00

Injection 2 — retired group duration difference
  Retired mean = 369.0s, Others mean = 232.3s

Injection 3 — age outlier cluster
  Clients aged 85+: 208


**Insight**

- Pearson r = 0.871 is a very strong correlation. The agent should detect this, name both columns, and report a value in this range.

- Retired clients average 369s vs 232s for everyone else, a difference of ~137 seconds, roughly 2.3 minutes. That's a large, meaningful effect. The agent should detect this group difference and run an appropriate test

- We see 208 clients aged 85+ rather than the 150 we injected. That means the original dataset already had ~58 clients in that age range, our injection added on top of them. The agent should flag this cluster. We'll score it as correct if it identifies anomalous concentration in the upper age range.



---



### Why a formal ground truth registry?

Keeping injections as loose notes or comments is not enough. Once we run the agent N=30 times, we need to score every run programmatically, that requires the ground truths to be structured data, not prose.

The registry serves three purposes:

1. **Scoring anchor** — every agent finding is compared against this dictionary. A finding either matches an entry or it doesn't.
2. **Tolerance bounds** — real LLM outputs won't report r=0.871 exactly. The registry defines acceptable ranges (e.g. r ± 0.05) so scoring is fair but not brittle.
3. **Audit trail** — anyone reading the notebook can see exactly what was injected, what test was expected, and what values are considered correct — before seeing any agent results.

### How the registry is structured

Each entry captures four things:

- **Type** — what kind of signal it is (correlation, group difference, outlier)
- **Columns** — which columns are involved
- **Expected test** — the statistically correct test for this signal and data type
- **Expected values + tolerance** — the numeric ground truth with an acceptable margin

This structure maps directly to the four evaluation metrics we will score later — statistical accuracy, hallucination rate, reasoning coherence, and test selection appropriateness.

In [11]:
# Formal ground truth registry
GROUND_TRUTHS = {
    "injection_1": {
        "type": "correlation",
        "col_a": "euribor3m",
        "col_b": "duration",
        "expected_test": "pearson",
        "expected_r": 0.871,
        "tolerance": 0.05
    },
    "injection_2": {
        "type": "group_difference",
        "col": "duration",
        "group_col": "job",
        "group_val": "retired",
        "expected_test": "mann_whitney",
        "expected_mean_retired": 369.0,
        "expected_mean_others": 232.3,
        "tolerance_seconds": 10
    },
    "injection_3": {
        "type": "outlier_cluster",
        "col": "age",
        "expected_test": "iqr_inspection",
        "threshold": 85,
        "expected_count": 208,
        "tolerance_count": 10
    }
}

print("=== Ground Truth Registry ===\n")
for k, v in GROUND_TRUTHS.items():
    col = v.get("col_a", v.get("col", ""))
    print(f"{k}: {v['type']} — {col} | expected test: {v['expected_test']}")

=== Ground Truth Registry ===

injection_1: correlation — euribor3m | expected test: pearson
injection_2: group_difference — duration | expected test: mann_whitney
injection_3: outlier_cluster — age | expected test: iqr_inspection


**Insight**

- Every agent run from this point is scored against exactly these three entries

- The test chosen, the columns identified, and the numeric values reported all have defined correct answers and tolerance bounds.



---



---



## Section 3: Agent Architecture

### Objective
Define the structure of the agent before writing a single line of agent code. Architecture decisions made here directly affect evaluation outcomes, a poorly designed tool schema or system prompt produces failures that look like model limitations but are actually engineering problems.

### How the agent works
The agent operates in a closed reasoning loop. At each turn it receives the dataset metadata and any previous tool results, selects one tool to call, observes the result, and decides what to do next. It continues until it calls the terminal tool `summarize_findings` or hits the maximum turn limit.



---



### Three core design decisions

**1. Metadata not raw data**
The agent never sees the raw dataframe. It receives a structured metadata object — column names, types, summary statistics, and null counts. This keeps prompts compact, reduces token cost, and forces the agent to request specific analyses via tool calls rather than eyeballing the data directly.

**2. Tools as guardrails**
Every analysis the agent can run is defined as a typed tool with a validated input schema. This prevents the agent from inventing analyses that don't exist or calling tests with incompatible column types. The tool layer is where statistical correctness is enforced — not the prompt.

**3. Structured output only**
The agent cannot emit free-form findings. All insights must be submitted via `summarize_findings` in a strict JSON schema. This makes the hallucination checker deterministic — every claim is machine-parseable and cross-referenceable against the tool call history.

### The five tools
| Tool | Purpose |
|------|---------|
| `get_column_stats` | Descriptive statistics for a single column |
| `check_assumptions` | Normality, variance, sample size checks before test selection |
| `run_statistical_test` | Execute a named statistical test, return p-value + effect size |
| `plot_distribution` | Request a distribution visualization |
| `summarize_findings` | Terminal tool — submit structured findings, end the loop |



---



## Metadata to Agent


### Why the agent receives metadata, not raw data

The agent does not receive the raw dataframe. It receives a structured metadata object instead. This is a deliberate architectural choice with three reasons:

**1. Token cost**
The full `df_injected` has 41,188 rows × 21 columns. Passing that directly to Claude would consume hundreds of thousands of tokens per run — expensive, slow, and unnecessary. The metadata object captures everything the agent needs to reason about the data in under 2,000 tokens.

**2. Forcing deliberate analysis**
If the agent could see the raw data, it could pattern-match on visible values and produce findings without running any tests. By restricting access to metadata only, every specific claim the agent makes must come from a tool call. This is what makes the hallucination checker possible — there is no legitimate path to a finding that bypasses the tool layer.

**3. Mimicking production reality**
In a real production pipeline, an agent monitoring a live dataset cannot load the full dataframe into a prompt. It queries a data warehouse via tools. This architecture mirrors that constraint — making the design realistic, not just functional for this notebook.

### How the metadata object is structured
For each column we capture: dtype, semantic type, null and unknown counts, and either summary statistics (for numeric columns) or top value frequencies (for categorical columns). This is everything a statistician would want to know before deciding which analysis to run.

In [13]:
# Build structured metadata object to pass to the agent
# The agent never sees the raw dataframe — only this object

def build_metadata(df):
    metadata = {}

    for col in df.columns:
        col_data = {}

        # Dtype
        col_data["dtype"] = str(df[col].dtype)

        # Semantic type
        if df[col].dtype in ["int64", "float64"]:
            col_data["semantic_type"] = "continuous"
        else:
            col_data["semantic_type"] = "categorical"

        # Null counts (both NaN and "unknown")
        col_data["null_count"] = int(df[col].isnull().sum())
        if df[col].dtype == "object":
            col_data["unknown_count"] = int((df[col] == "unknown").sum())

        # Summary stats
        if df[col].dtype in ["int64", "float64"]:
            col_data["mean"]   = round(float(df[col].mean()), 2)
            col_data["std"]    = round(float(df[col].std()), 2)
            col_data["min"]    = round(float(df[col].min()), 2)
            col_data["max"]    = round(float(df[col].max()), 2)
            col_data["median"] = round(float(df[col].median()), 2)
        else:
            col_data["unique_values"] = int(df[col].nunique())
            col_data["top_values"] = df[col].value_counts().head(3).to_dict()

        metadata[col] = col_data

    return metadata

metadata = build_metadata(df_injected)

print("=== Metadata object built ===\n")
print(f"Total columns profiled: {len(metadata)}")
print(f"\nSample — 'age':\n{json.dumps(metadata['age'], indent=2)}")
print(f"\nSample — 'job':\n{json.dumps(metadata['job'], indent=2)}")

=== Metadata object built ===

Total columns profiled: 21

Sample — 'age':
{
  "dtype": "int64",
  "semantic_type": "continuous",
  "null_count": 0,
  "mean": 40.21,
  "std": 10.87,
  "min": 17.0,
  "max": 98.0,
  "median": 38.0
}

Sample — 'job':
{
  "dtype": "object",
  "semantic_type": "categorical",
  "null_count": 0,
  "unknown_count": 330,
  "unique_values": 12,
  "top_values": {
    "admin.": 10422,
    "blue-collar": 9254,
    "technician": 6743
  }
}


**Insight**

- Age looks correct and matche searlier statistical analysis

- Job capture 'Unknown' entries we identified earlier. Otherwise it would have been a failure in capturing the hidden missingness.

- The metadata object is doing its job, it's compact, structured, and contains the right signals. All 21 columns are profiled and ready to pass to the agent.



---



---



### Tool schema design

Each tool is defined with a strict input schema. This is not just documentation — Claude uses the schema to validate its own tool calls before executing them. A poorly defined schema is an open door for test misapplication.

Three rules guided every tool design decision:

**1. Inputs are typed and constrained**
Every parameter has an explicit type and where possible an enum of allowed values. The `run_statistical_test` tool does not accept a free-form test name — it accepts only named tests from a controlled list. This prevents the agent from inventing a test that doesn't exist.

**2. Tools return structured JSON, never prose**
Every tool returns a dictionary. This keeps results machine-parseable for the hallucination checker and evaluation scorer. A tool that returns a sentence cannot be scored deterministically.

**3. `summarize_findings` is the only exit**
The agent cannot end the loop by stopping — it must call `summarize_findings` explicitly. This forces every run to produce a structured output regardless of how the reasoning went, and gives us a consistent artifact to score across all N=30 runs.

In [14]:
# Define the five tool schemas
# These are passed to Claude on every API call

tools = [
    {
        "name": "get_column_stats",
        "description": "Get descriptive statistics for a single column. Always call this before running any statistical test on a column.",
        "input_schema": {
            "type": "object",
            "properties": {
                "column": {
                    "type": "string",
                    "description": "Column name to analyze"
                },
                "stat_type": {
                    "type": "string",
                    "enum": ["distribution", "outliers", "frequency"],
                    "description": "Type of statistics to return"
                }
            },
            "required": ["column", "stat_type"]
        }
    },
    {
        "name": "check_assumptions",
        "description": "Check statistical assumptions before selecting a test. Always call this before run_statistical_test.",
        "input_schema": {
            "type": "object",
            "properties": {
                "column": {
                    "type": "string",
                    "description": "Column name to check"
                },
                "test_type": {
                    "type": "string",
                    "enum": ["normality", "variance_homogeneity", "sample_size"],
                    "description": "Assumption to check"
                }
            },
            "required": ["column", "test_type"]
        }
    },
    {
        "name": "run_statistical_test",
        "description": "Execute a statistical test. Must call check_assumptions first. Returns p-value, statistic, and effect size.",
        "input_schema": {
            "type": "object",
            "properties": {
                "test": {
                    "type": "string",
                    "enum": ["pearson", "spearman", "mann_whitney", "chi_square", "t_test", "iqr_inspection"],
                    "description": "Statistical test to run"
                },
                "col_a": {
                    "type": "string",
                    "description": "First column"
                },
                "col_b": {
                    "type": "string",
                    "description": "Second column or group column"
                },
                "group_val": {
                    "type": "string",
                    "description": "For group comparisons — the group value to isolate (e.g. 'retired')"
                }
            },
            "required": ["test", "col_a", "col_b"]
        }
    },
    {
        "name": "plot_distribution",
        "description": "Generate a distribution plot for a column, optionally grouped by a categorical variable.",
        "input_schema": {
            "type": "object",
            "properties": {
                "column": {
                    "type": "string",
                    "description": "Column to plot"
                },
                "plot_type": {
                    "type": "string",
                    "enum": ["histogram", "boxplot", "countplot"],
                    "description": "Type of plot"
                },
                "group_by": {
                    "type": "string",
                    "description": "Optional column to group by"
                }
            },
            "required": ["column", "plot_type"]
        }
    },
    {
        "name": "summarize_findings",
        "description": "Terminal tool. Submit all findings in structured format and end the analysis loop. Only call this when analysis is complete.",
        "input_schema": {
            "type": "object",
            "properties": {
                "findings": {
                    "type": "array",
                    "description": "List of findings",
                    "items": {
                        "type": "object",
                        "properties": {
                            "finding": {"type": "string"},
                            "supporting_test": {"type": "string"},
                            "test_result": {"type": "object"},
                            "confidence": {
                                "type": "string",
                                "enum": ["Low", "Medium", "High"]
                            },
                            "implication": {"type": "string"}
                        },
                        "required": ["finding", "supporting_test", "test_result", "confidence", "implication"]
                    }
                }
            },
            "required": ["findings"]
        }
    }
]

print(f"Tools defined: {len(tools)}")
for t in tools:
    print(f"  - {t['name']}")

Tools defined: 5
  - get_column_stats
  - check_assumptions
  - run_statistical_test
  - plot_distribution
  - summarize_findings


### What this tool schema achieves

The schema is doing three jobs simultaneously: guiding the agent's reasoning order, enforcing statistical guardrails, and producing output the evaluation layer can score deterministically.

**Guided reasoning order**
`get_column_stats` and `check_assumptions` are described as prerequisites to `run_statistical_test`. This is not hard enforcement — Claude can skip them — but when it does, that skip is a scoreable failure. The schema sets the expected path; deviations from it are signal, not noise.

**Statistical guardrails**
`run_statistical_test` accepts only six named tests via enum. The agent cannot invent a test that isn't on the list. This is where statistical correctness is enforced at the architecture level — not left to the prompt.

**Evaluation-ready output**
`summarize_findings` requires five fields per finding: `finding`, `supporting_test`, `test_result`, `confidence`, and `implication`. All are mandatory. A finding without a `supporting_test` is structurally rejected. This is what makes the hallucination checker deterministic — every claim is machine-parseable and traceable back to a specific tool call.

The schema is not just an interface definition. It is the first layer of the evaluation framework.



---



## Tool Execution Engine


In [18]:
# Tool execution engine
# These are the Python functions that run when Claude calls each tool

def execute_tool(tool_name, tool_input, df):

    # --- Tool 1: get_column_stats ---
    if tool_name == "get_column_stats":
        col = tool_input["column"]
        stat_type = tool_input["stat_type"]

        if col not in df.columns:
            return {"error": f"Column '{col}' not found"}

        if stat_type == "distribution":
            return {
                "column": col,
                "mean":   round(float(df[col].mean()), 3) if df[col].dtype != "object" else None,
                "median": round(float(df[col].median()), 3) if df[col].dtype != "object" else None,
                "std":    round(float(df[col].std()), 3) if df[col].dtype != "object" else None,
                "min":    round(float(df[col].min()), 3) if df[col].dtype != "object" else None,
                "max":    round(float(df[col].max()), 3) if df[col].dtype != "object" else None,
                "skewness": round(float(df[col].skew()), 3) if df[col].dtype != "object" else None
            }

        elif stat_type == "outliers":
            q1  = df[col].quantile(0.25)
            q3  = df[col].quantile(0.75)
            iqr = q3 - q1
            outliers = df[(df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)]
            return {
                "column": col,
                "q1": round(float(q1), 3),
                "q3": round(float(q3), 3),
                "iqr": round(float(iqr), 3),
                "lower_fence": round(float(q1 - 1.5 * iqr), 3),
                "upper_fence": round(float(q3 + 1.5 * iqr), 3),
                "outlier_count": int(len(outliers)),
                "outlier_pct": round(len(outliers) / len(df) * 100, 2)
            }

        elif stat_type == "frequency":
            counts = df[col].value_counts().head(10).to_dict()
            return {
                "column": col,
                "unique_count": int(df[col].nunique()),
                "unknown_count": int((df[col] == "unknown").sum()),
                "top_10": counts
            }

    # --- Tool 2: check_assumptions ---
    elif tool_name == "check_assumptions":
        col = tool_input["column"]
        test_type = tool_input["test_type"]

        if test_type == "normality":
            # Shapiro-Wilk on a sample (max 5000 rows)
            sample = df[col].dropna().sample(min(5000, len(df)), random_state=42)
print("Tool execution engine ready")

Tool execution engine ready
